# LeetCode 3498 — Reverse Degree of a String bằng Python

Link đề: https://leetcode.com/problems/reverse-degree-of-a-string/ · Độ khó: **Easy** · Chủ đề: String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def reverseDegree(self, s: str) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi ký tự có một **thứ hạng ngược** trong bảng chữ cái (`'a'` = 26, `'b'` = 25, ..., `'z'` = 1) và được nhân với **vị trí (đếm từ 1)** trong chuỗi, rồi cộng lại.
2. Biết đổi ký tự thành số bằng `ord`, bằng chuỗi tra cứu `"zyx...a"`, hoặc bằng `dict`.
3. Biết dùng `enumerate(s, start=1)`, `sum` với generator, và `itertools.accumulate`.
4. Viết 4 cách: vòng lặp với `ord`, tra cứu bằng chuỗi đảo ngược, `sum(...)` một dòng, và cách **tổng hậu tố** (suffix sums) — một góc nhìn khác của cùng công thức.
5. Tránh các lỗi: đánh số vị trí từ `0`, tính thứ hạng thuận (`'a'` = 1), quên nhân vị trí, và sai lệch một đơn vị khi đổi `ord` sang thứ hạng ngược.

## 2. Đề bài & ví dụ

Cho chuỗi `s` gồm các chữ cái thường. **Reverse degree** của `s` được tính như sau:

1. Với mỗi ký tự, lấy **thứ hạng của nó trong bảng chữ cái đảo ngược** (`'a'` -> `26`, `'b'` -> `25`, ..., `'z'` -> `1`).
2. Nhân thứ hạng đó với **vị trí của ký tự trong chuỗi, đếm từ `1`**.
3. Cộng tất cả các tích.

Trả về reverse degree của `s`.

| Ví dụ | s | Output | Giải thích |
|---|---|---|---|
| 1 | `"abc"` | `148` | `'a'`: `26·1 = 26`; `'b'`: `25·2 = 50`; `'c'`: `24·3 = 72`; tổng `148` |
| 2 | `"zaza"` | `160` | `1·1 + 26·2 + 1·3 + 26·4 = 1 + 52 + 3 + 104 = 160` |

Constraints: `1 <= s.length <= 1000`; `s` chỉ gồm chữ cái tiếng Anh thường.

In [1]:
examples = [("abc", 148), ("zaza", 160)]
alphabet_reversed = "zyxwvutsrqponmlkjihgfedcba"
for s, expected in examples:
    total = 0
    for pos, ch in enumerate(s, start=1):
        rank = alphabet_reversed.index(ch) + 1     # 'z' -> 1, 'a' -> 26
        total += rank * pos
        print(f"  {ch}: hạng ngược {rank} × vị trí {pos} = {rank * pos}")
    print(f"s={s!r}: tổng {total} (mong đợi {expected})")

  a: hạng ngược 26 × vị trí 1 = 26
  b: hạng ngược 25 × vị trí 2 = 50
  c: hạng ngược 24 × vị trí 3 = 72
s='abc': tổng 148 (mong đợi 148)
  z: hạng ngược 1 × vị trí 1 = 1
  a: hạng ngược 26 × vị trí 2 = 52
  z: hạng ngược 1 × vị trí 3 = 3
  a: hạng ngược 26 × vị trí 4 = 104
s='zaza': tổng 160 (mong đợi 160)


## 3. Kiến thức nền

### 3.1. `ord` và mã ký tự

`ord('a')` là `97`, `ord('b')` là `98`, ..., `ord('z')` là `122`. Với chữ thường, `ord(ch) - ord('a')` cho **chỉ số thuận** `0..25`. Chỉ số này cộng `1` ra thứ hạng thuận (`'a'` = 1); còn thứ hạng **ngược** là `26 - (ord(ch) - ord('a'))`, rút gọn thành `123 - ord(ch)` vì `26 + 97 = 123`.

In [2]:
for ch in "azm":
    forward = ord(ch) - ord('a') + 1
    reverse = 26 - (ord(ch) - ord('a'))
    print(ch, ord(ch), "thuận", forward, "ngược", reverse, "| 123 - ord =", 123 - ord(ch))

a 97 thuận 1 ngược 26 | 123 - ord = 26
z 122 thuận 26 ngược 1 | 123 - ord = 1
m 109 thuận 13 ngược 14 | 123 - ord = 14


### 3.2. `enumerate(s, start=1)` — vị trí đếm từ 1

`enumerate(s)` mặc định bắt đầu từ `0`; đề yêu cầu vị trí **từ `1`** nên dùng tham số `start=1`. Đây là chỗ rất hay sai một đơn vị.

In [3]:
print(list(enumerate("abc")))
print(list(enumerate("abc", start=1)))

[(0, 'a'), (1, 'b'), (2, 'c')]
[(1, 'a'), (2, 'b'), (3, 'c')]


### 3.3. Tra cứu bằng chuỗi hoặc `dict`

`"zyxw...a".index(ch) + 1` cho thứ hạng ngược (vị trí của `ch` trong chuỗi đảo ngược, cộng `1`). Hoặc dựng `dict` một lần: `{ch: 26 - i for i, ch in enumerate("abcdefghijklmnopqrstuvwxyz")}`. Tra `dict` là `O(1)`; `.index` duyệt tối đa `26` ký tự (cũng là hằng số).

In [4]:
import string
rank = {ch: 26 - i for i, ch in enumerate(string.ascii_lowercase)}
print(rank["a"], rank["b"], rank["z"])
print(string.ascii_lowercase[::-1])        # cắt lát bước -1 đảo ngược chuỗi

26 25 1
zyxwvutsrqponmlkjihgfedcba


### 3.4. `sum` với generator

`sum(f(x) for x in iterable)` cộng các giá trị mà không tạo danh sách trung gian. Kết hợp với `enumerate` ta viết cả bài trong một biểu thức.

In [5]:
s = "abc"
print(sum(pos * (123 - ord(ch)) for pos, ch in enumerate(s, start=1)))

148


### 3.5. `itertools.accumulate`

`accumulate(iterable)` trả về **tổng tích lũy**: `accumulate([1, 2, 3])` cho `1, 3, 6`. Ta dùng nó ở Cách 4 để biểu diễn `Σ pos·rank` thành tổng của các **tổng hậu tố**.

In [6]:
from itertools import accumulate
print(list(accumulate([1, 2, 3])))
print(list(accumulate(reversed([26, 25, 24]))))     # tổng hậu tố, tính từ cuối

[1, 3, 6]
[24, 49, 75]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay `s = "zaza"`:

```
vị trí : 1    2    3    4
ký tự  : z    a    z    a
hạng   : 1    26   1    26
tích   : 1    52   3    104        -> tổng 160
```

Bài này là **mô phỏng thẳng**: đọc từng ký tự một lần, tính `thứ_hạng × vị_trí`, cộng dồn. Không có bẫy thuật toán; độ khó nằm ở chỗ **đổi ký tự sang thứ hạng ngược đúng** và **đếm vị trí từ 1**.

Ước lượng cỡ kết quả: tối đa `26 · (1 + 2 + ... + 1000) = 26 · 500 500 = 13 013 000` nên vừa trong số nguyên Python (và cả `int` 32-bit).

## 5. Cách 1 — Brute Force / baseline

Duyệt chỉ số `i` từ `0` đến `n-1`, thứ hạng ngược là `26 - (ord(s[i]) - ord('a'))`, vị trí (từ 1) là `i + 1`; cộng dồn vào `total`.

In [7]:
class SolutionLoop:
    def reverseDegree(self, s):
        total = 0
        for i in range(len(s)):
            rank = 26 - (ord(s[i]) - ord('a'))
            total += rank * (i + 1)
        return total

print(SolutionLoop().reverseDegree("abc"))
print(SolutionLoop().reverseDegree("zaza"))

148
160


**Độ phức tạp:** `O(n)` thời gian (một lần duyệt), `O(1)` bộ nhớ thêm. Đây đã là tối ưu về bậc (phải đọc từng ký tự ít nhất một lần).

## 6. Ý tưởng thuật toán / bước đột phá

Vì bài đã là `O(n)` tối ưu, "bước đột phá" ở đây là **các cách nhìn công thức** cho gọn và ít sai:

- **Tra cứu thay vì tính:** thứ hạng ngược của `ch` chính là vị trí của `ch` trong chuỗi `"zyx...a"` cộng `1`. Không cần nhớ `26`, `97`, `123`.
- **Gộp vào một biểu thức:** `sum(i * rank(ch) for i, ch in enumerate(s, 1))`.
- **Tổng hậu tố:** gọi `r_j` là thứ hạng của ký tự thứ `j` (từ `1`). Ta có `Σ_j j·r_j = Σ_{t=1..n} (r_t + r_{t+1} + ... + r_n)`, vì `r_j` xuất hiện trong đúng `j` tổng hậu tố đầu tiên (các hậu tố bắt đầu từ `t = 1..j`). Vậy **reverse degree = tổng của các tổng hậu tố**.

**Quy tắc "nếu… thì…":**

- Nếu ký tự là `'a'` thì thứ hạng là `26`, nếu `'z'` thì `1`; hạng + hạng thuận luôn bằng `27`.
- Vị trí luôn đếm từ `1`, nên ký tự cuối được nhân với `n`.

In [8]:
s = "abc"
r = [123 - ord(ch) for ch in s]
suffix = list(accumulate(reversed(r)))[::-1]      # tổng hậu tố bắt đầu tại mỗi vị trí
print("thứ hạng :", r)
print("hậu tố   :", suffix, "-> tổng", sum(suffix))
print("trực tiếp:", sum(pos * rk for pos, rk in enumerate(r, start=1)))

thứ hạng : [26, 25, 24]
hậu tố   : [75, 49, 24] -> tổng 148
trực tiếp: 148


## 7. Các cách tiếp theo

Ta có 4 cách: (1) vòng lặp với `ord`; (2) tra cứu bằng chuỗi đảo ngược; (3) `sum` một dòng với `enumerate(start=1)`; (4) tổng của các tổng hậu tố.

### Cách 2 — Tra cứu bằng chuỗi đảo ngược

Dựng `rev = "zyxwvutsrqponmlkjihgfedcba"` (hoặc `string.ascii_lowercase[::-1]`). Thứ hạng ngược của `ch` là `rev.index(ch) + 1`. Dễ đọc, khó sai.

In [9]:
import string

class SolutionLookup:
    def reverseDegree(self, s):
        rev = string.ascii_lowercase[::-1]       # 'z' ở vị trí 0 -> hạng 1
        total = 0
        for pos, ch in enumerate(s, start=1):
            total += pos * (rev.index(ch) + 1)
        return total

print(SolutionLookup().reverseDegree("abc"))
print(SolutionLookup().reverseDegree("zaza"))

148
160


Độ phức tạp Cách 2: `index` duyệt tối đa `26` ký tự -> `O(26 · n) = O(n)` thời gian, `O(1)` bộ nhớ. Dùng `dict` thì mỗi tra cứu `O(1)`.

### Cách 3 — Một dòng với `sum` và `enumerate`

Viết cả công thức thành một biểu thức generator: `sum(pos * (123 - ord(ch)) for pos, ch in enumerate(s, start=1))`.

In [10]:
class SolutionOneLine:
    def reverseDegree(self, s):
        return sum(pos * (123 - ord(ch)) for pos, ch in enumerate(s, start=1))

print(SolutionOneLine().reverseDegree("abc"))
print(SolutionOneLine().reverseDegree("zaza"))

148
160


Độ phức tạp Cách 3: `O(n)` thời gian, `O(1)` bộ nhớ (generator).

### Cách 4 — Tổng của các tổng hậu tố

Tính danh sách thứ hạng `r`, duyệt **từ cuối về đầu** giữ tổng chạy `run`; mỗi lần cộng `run` (là tổng hậu tố bắt đầu tại vị trí đó) vào kết quả. Theo mục 6 kết quả bằng `Σ j · r_j`.

In [11]:
class SolutionSuffix:
    def reverseDegree(self, s):
        total = run = 0
        for ch in reversed(s):
            run += 123 - ord(ch)       # run = tổng thứ hạng từ vị trí hiện tại đến cuối
            total += run               # cộng tổng hậu tố
        return total

print(SolutionSuffix().reverseDegree("abc"))
print(SolutionSuffix().reverseDegree("zaza"))

148
160


Độ phức tạp Cách 4: `O(n)` thời gian, `O(1)` bộ nhớ. Hữu ích khi cần cập nhật khi chuỗi thay đổi hoặc tính nhiều đoạn bằng tổng tiền tố.

## 8. TRACE từng bước

Trace Cách 1 với `s = "zaza"`:

| bước | `i` | `s[i]` | `ord - 97` | `rank = 26 - ...` | vị trí `i+1` | tích | `total` |
|---|---|---|---|---|---|---|---|
| 1 | 0 | `z` | 25 | 1 | 1 | 1 | 1 |
| 2 | 1 | `a` | 0 | 26 | 2 | 52 | 53 |
| 3 | 2 | `z` | 25 | 1 | 3 | 3 | 56 |
| 4 | 3 | `a` | 0 | 26 | 4 | 104 | 160 |

Kết quả `160`.

In [12]:
def trace_reverse_degree(s):
    total = 0
    print(f"s = {s!r}")
    for i in range(len(s)):
        idx = ord(s[i]) - ord('a')
        rank = 26 - idx
        pos = i + 1
        total += rank * pos
        print(f"bước {i + 1}: ký tự {s[i]!r} idx={idx:>2} rank={rank:>2} pos={pos} tích={rank * pos:>4} total={total}")
    return total

assert trace_reverse_degree("zaza") == 160
assert trace_reverse_degree("abc") == 148

s = 'zaza'
bước 1: ký tự 'z' idx=25 rank= 1 pos=1 tích=   1 total=1
bước 2: ký tự 'a' idx= 0 rank=26 pos=2 tích=  52 total=53
bước 3: ký tự 'z' idx=25 rank= 1 pos=3 tích=   3 total=56
bước 4: ký tự 'a' idx= 0 rank=26 pos=4 tích= 104 total=160
s = 'abc'
bước 1: ký tự 'a' idx= 0 rank=26 pos=1 tích=  26 total=26
bước 2: ký tự 'b' idx= 1 rank=25 pos=2 tích=  50 total=76
bước 3: ký tự 'c' idx= 2 rank=24 pos=3 tích=  72 total=148


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def reverseDegree(self, s: str) -> int:
        total = 0
        for pos, ch in enumerate(s, start=1):
            total += pos * (123 - ord(ch))      # 123 - ord(ch) = 26 - (ord(ch) - ord('a'))
        return total

print(Solution().reverseDegree("abc"), Solution().reverseDegree("zaza"))

148 160


## 10. Vì sao thuật toán đúng

- **Thứ hạng ngược:** với `idx = ord(ch) - ord('a') ∈ [0, 25]`, thứ hạng ngược là `26 - idx` (`'a'` -> `26`, `'z'` -> `1`). Vì `ord('a') = 97`, `26 - (ord(ch) - 97) = 123 - ord(ch)`, đúng như Cách 3 và code nộp.
- **Vị trí từ 1:** `enumerate(..., start=1)` hoặc `i + 1` cho vị trí 1-indexed theo đề.
- **Tổng:** mỗi ký tự đóng góp đúng một tích, vòng lặp đi qua mỗi ký tự đúng một lần nên không thiếu không thừa.
- **Cách 4:** `r_j` thuộc các tổng hậu tố bắt đầu tại `t = 1, ..., j`, tức `j` lần; vậy tổng các tổng hậu tố bằng `Σ j · r_j`.
- **Kiểm chứng:** hai tính chất đóng: chuỗi toàn `'a'` độ dài `n` cho `26 · n(n+1)/2`; chuỗi toàn `'z'` cho `n(n+1)/2`. Cả hai được assert trong bộ test.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(s)`.

| Cách | Time | Space |
|---|---|---|
| 1. Vòng lặp với `ord` | `O(n)` | `O(1)` |
| 2. Tra cứu bằng chuỗi đảo ngược | `O(26 · n) = O(n)` | `O(1)` |
| 3. `sum` một dòng | `O(n)` | `O(1)` |
| 4. Tổng hậu tố | `O(n)` | `O(1)` |

Mọi cách đều tối ưu về bậc; khác nhau ở độ dễ đọc. Với `n <= 1000` chạy dưới một phần nghìn giây.

## 12. Bộ test

Test 1: ví dụ đề và các edge case (`n = 1`, toàn `a`, toàn `z`, độ dài `1000`). Mọi input tuân thủ `1 <= n <= 1000`, chỉ chữ cái thường.

In [14]:
impls = [SolutionLoop(), SolutionLookup(), SolutionOneLine(), SolutionSuffix(), Solution()]
tests = [
    ("abc", 148),
    ("zaza", 160),
    ("a", 26),                              # n = 1, hạng lớn nhất
    ("z", 1),                               # n = 1, hạng nhỏ nhất
    ("az", 26 * 1 + 1 * 2),                 # 28
    ("za", 1 * 1 + 26 * 2),                 # 53, đổi thứ tự thì khác
    ("m", 14),                              # 'm' là chữ thứ 13 -> hạng ngược 14
    ("n", 13),
    ("a" * 1000, 26 * 1000 * 1001 // 2),    # biên trên n, toàn 'a'
    ("z" * 1000, 1000 * 1001 // 2),         # biên trên n, toàn 'z'
]
for s, exp in tests:
    assert 1 <= len(s) <= 1000 and s.islower() and s.isalpha()
    for im in impls:
        got = im.reverseDegree(s)
        assert got == exp, (type(im).__name__, s[:10], got, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 10


Test 2: ngẫu nhiên có seed. Oracle độc lập: dựng `dict` thứ hạng ngược bằng vòng lặp rồi cộng; so với cả năm cài đặt.

In [15]:
import random
import string
random.seed(3498)

def oracle(s):
    rank = {}
    for i, ch in enumerate(string.ascii_lowercase):
        rank[ch] = 26 - i
    total = 0
    for j in range(1, len(s) + 1):
        total += j * rank[s[j - 1]]
    return total

for t in range(2000):
    n = random.randint(1, 1000) if t % 10 == 0 else random.randint(1, 30)
    pool = random.choice(["ab", "az", string.ascii_lowercase])
    s = "".join(random.choice(pool) for _ in range(n))
    assert 1 <= len(s) <= 1000 and s.islower()
    ref = oracle(s)
    for im in impls:
        assert im.reverseDegree(s) == ref, (type(im).__name__, s[:20])
print("2000 test ngẫu nhiên OK")

2000 test ngẫu nhiên OK


Test 3: tính chất. (a) Thêm một ký tự `ch` vào **cuối** `s` làm tăng kết quả đúng `(n+1)·rank(ch)`. (b) Cộng thứ hạng thuận và thứ hạng ngược của cùng ký tự luôn bằng `27`, nên `forward + reverse = 27 · n(n+1)/2`.

In [16]:
random.seed(98)
def forward_degree(s):
    return sum(pos * (ord(ch) - 96) for pos, ch in enumerate(s, start=1))

for _ in range(500):
    n = random.randint(1, 200)
    s = "".join(random.choice(string.ascii_lowercase) for _ in range(n))
    ch = random.choice(string.ascii_lowercase)
    assert Solution().reverseDegree(s + ch) == Solution().reverseDegree(s) + (n + 1) * (123 - ord(ch))
    assert forward_degree(s) + Solution().reverseDegree(s) == 27 * n * (n + 1) // 2
print("tính chất (thêm ký tự cuối, thuận + ngược = 27·n(n+1)/2): OK")

tính chất (thêm ký tự cuối, thuận + ngược = 27·n(n+1)/2): OK


Test 4: **vét cạn** toàn bộ chuỗi gồm 1 đến 3 chữ cái trong bảng 26 chữ (`26 + 676 + 17576 = 18278` chuỗi) và so cả năm cài đặt với oracle. Ngoài ra kiểm tra công thức đóng cho chuỗi **một chữ lặp lại**: `ch * n` có reverse degree `rank(ch) · n(n+1)/2`, vì mọi ký tự cùng một thứ hạng nên tổng là `rank` nhân tổng các vị trí `1 + 2 + ... + n`.

In [17]:
from itertools import product
total_checked = 0
for L in (1, 2, 3):
    for tup in product(string.ascii_lowercase, repeat=L):
        s = "".join(tup)
        ref = oracle(s)
        for im in impls:
            assert im.reverseDegree(s) == ref, (type(im).__name__, s)
        total_checked += 1
print("vét cạn", total_checked, "chuỗi độ dài 1..3 OK")

for ch in string.ascii_lowercase:
    for n in (1, 2, 7, 1000):
        assert Solution().reverseDegree(ch * n) == (123 - ord(ch)) * n * (n + 1) // 2
print("công thức chuỗi lặp một chữ OK")

vét cạn 18278 chuỗi độ dài 1..3 OK
công thức chuỗi lặp một chữ OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: đánh số vị trí từ `0`

In [18]:
def wrong_zero_index(s):
    total = 0
    for pos, ch in enumerate(s):                  # SAI: pos bắt đầu từ 0
        total += pos * (123 - ord(ch))
    return total

s = "abc"
print("sai :", wrong_zero_index(s))
print("đúng:", Solution().reverseDegree(s))
assert wrong_zero_index(s) != Solution().reverseDegree(s)

sai : 73
đúng: 148


Vì sao sai: ký tự đầu bị nhân với `0` nên mất hẳn; ra `0·26 + 1·25 + 2·24 = 73` thay vì `148`. **Sửa:** `enumerate(s, start=1)` hoặc `i + 1`.

### Lỗi 2: tính thứ hạng thuận (`'a'` = 1) thay vì ngược

In [19]:
def wrong_forward_rank(s):
    total = 0
    for pos, ch in enumerate(s, start=1):
        total += pos * (ord(ch) - ord('a') + 1)   # SAI: 'a' = 1, 'z' = 26
    return total

s = "abc"
print("sai :", wrong_forward_rank(s))
print("đúng:", Solution().reverseDegree(s))
assert wrong_forward_rank(s) != Solution().reverseDegree(s)

sai : 14
đúng: 148


Vì sao sai: đề dùng **bảng chữ cái đảo ngược** (`'a'` = 26). Code sai ra `1·1 + 2·2 + 3·3 = 14`. **Sửa:** `26 - (ord(ch) - ord('a'))`.

### Lỗi 3: quên nhân với vị trí

In [20]:
def wrong_no_position(s):
    return sum(123 - ord(ch) for ch in s)         # SAI: thiếu nhân với vị trí

s = "zaza"
print("sai :", wrong_no_position(s))
print("đúng:", Solution().reverseDegree(s))
assert wrong_no_position(s) != Solution().reverseDegree(s)

sai : 54
đúng: 160


Vì sao sai: ra `1 + 26 + 1 + 26 = 54` thay vì `160`; kết quả cũng không còn phụ thuộc thứ tự ký tự (`"az"` và `"za"` cho cùng một giá trị). **Sửa:** nhân với vị trí 1-indexed.

### Lỗi 4: sai lệch một đơn vị khi đổi `ord` sang thứ hạng ngược

In [21]:
def wrong_off_by_one(s):
    total = 0
    for pos, ch in enumerate(s, start=1):
        total += pos * (122 - ord(ch))            # SAI: 122 - ord(ch) cho 'z' = 0, 'a' = 25
    return total

s = "zaza"
print("sai :", wrong_off_by_one(s))
print("đúng:", Solution().reverseDegree(s))
assert wrong_off_by_one(s) != Solution().reverseDegree(s)

sai : 150
đúng: 160


Vì sao sai: `'z'` phải có hạng `1` chứ không phải `0`. Hằng số đúng là `123` (vì `26 + ord('a') = 123`), hoặc tra bằng `rev.index(ch) + 1` để khỏi nhớ hằng số. **Sửa:** `123 - ord(ch)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "gán trọng số cho ký tự rồi cộng" -> mô phỏng một lần duyệt, `O(n)`.
- Đổi chữ thường sang số: `ord(ch) - ord('a')` (chỉ số thuận `0..25`); thứ hạng ngược: `26 - chỉ_số`, hoặc dùng chuỗi `string.ascii_lowercase[::-1]`.
- Hai chữ "1-indexed" trong đề -> `enumerate(s, start=1)`.
- Kiểm tra nhanh bằng tay trên ví dụ nhỏ `"abc"` -> `148`, `"zaza"` -> `160`.
- Dùng hai chuỗi `abc..z` và `zyx..a` ghép đôi cho nhanh; tổng hai thứ hạng luôn `27`.
- Bài họ hàng (mô phỏng chuỗi theo chữ cái): `1832. Check if the Sentence Is Pangram`, `2325. Decode the Message`.

## 15. Biến thể của bài

1. **Thứ hạng thuận:** `'a'` = 1, ..., `'z'` = 26 (Lỗi 2 trở thành đáp án đúng).
2. **Đánh số vị trí từ `0`**, hoặc nhân với **vị trí tính từ cuối** thay vì từ đầu.
3. **Chữ hoa lẫn chữ thường:** quy đổi bằng `ch.lower()` trước.
4. **Nhiều truy vấn đoạn `[l, r]`:** dùng tổng tiền tố của `r` và của `j·r` để trả lời `O(1)` mỗi truy vấn.
5. **Tìm ký tự đóng góp nhiều nhất** hoặc **vị trí làm tổng lớn nhất** khi được đổi một ký tự.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vòng lặp với `ord` | tính từng tích rồi cộng | `O(n)` | `O(1)` | rõ ràng, dễ debug |
| 2. Tra cứu chuỗi đảo ngược | `rev.index(ch) + 1` | `O(26n)` | `O(1)` | khỏi nhớ hằng số |
| 3. `sum` một dòng | generator + `enumerate` | `O(n)` | `O(1)` | **gọn, nên nộp** |
| 4. Tổng hậu tố | tổng của các tổng hậu tố | `O(n)` | `O(1)` | hiểu cấu trúc, mở rộng truy vấn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `reverseDegree("az")` bằng bao nhiêu?

**Bài 2 (tính tay):** tính reverse degree của `"ba"` bằng bảng `ký tự / hạng / vị trí / tích`.

**Bài 3 (điền TODO):** hoàn thành `rev_rank(ch)` trả về thứ hạng ngược của một chữ cái thường.

**Bài 4 (tự viết lại):** viết `forward_degree(s)` tính **thứ hạng thuận** (`'a'` = 1) nhân vị trí từ 1, rồi cộng.

**Bài 5 (mini challenge):** viết `prefix_degrees(s)` trả về danh sách reverse degree của **mọi tiền tố** `s[:1], s[:2], ..., s[:n]` trong `O(n)` (đừng tính lại từ đầu mỗi lần).

In [22]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().reverseDegree("az"))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def rev_rank(ch):
    # TODO: trả về 26 với 'a', 1 với 'z'
    return 0

28


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** mỗi ký tự đóng góp `hạng ngược × vị trí (từ 1)`; `'a'` hạng `26`, `'z'` hạng `1`.

**Mức 2 (vừa):** Bài 3: `26 - (ord(ch) - ord('a'))`. Bài 4: thứ hạng thuận là `ord(ch) - ord('a') + 1`. Bài 5: giữ `total` chạy; ở bước `pos` cộng thêm `pos * rank`, rồi `append(total)`.

**Mức 3 (gần đáp án):** Bài 1: `26·1 + 1·2 = 28`. Bài 2: `'b'` hạng `25` vị trí `1` -> `25`; `'a'` hạng `26` vị trí `2` -> `52`; tổng `77`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
assert Solution().reverseDegree("az") == 28

# Bài 2: 25*1 + 26*2 = 77
assert Solution().reverseDegree("ba") == 77

# Bài 3
def rev_rank(ch):
    return 26 - (ord(ch) - ord('a'))
assert rev_rank('a') == 26 and rev_rank('z') == 1 and rev_rank('m') == 14

# Bài 4
def forward_degree_ex(s):
    total = 0
    for pos, ch in enumerate(s, start=1):
        total += pos * (ord(ch) - ord('a') + 1)
    return total
assert forward_degree_ex("abc") == 14                 # 1*1 + 2*2 + 3*3
assert forward_degree_ex("za") == 26 * 1 + 1 * 2

# Bài 5
def prefix_degrees(s):
    out, total = [], 0
    for pos, ch in enumerate(s, start=1):
        total += pos * rev_rank(ch)
        out.append(total)
    return out
assert prefix_degrees("abc") == [26, 76, 148]
assert prefix_degrees("zaza")[-1] == 160
assert all(prefix_degrees("zaza")[i] == Solution().reverseDegree("zaza"[:i + 1]) for i in range(4))
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "gán trọng số theo ký tự và vị trí":**

```python
total = 0
for pos, ch in enumerate(s, start=1):      # vị trí từ 1
    total += pos * (123 - ord(ch))         # hạng ngược: 'a' -> 26, 'z' -> 1
return total
```

**Checklist trước khi submit:**

- [ ] Vị trí đếm từ `1` (`start=1`)?
- [ ] Thứ hạng **ngược**: `'a'` = 26, `'z'` = 1 (hằng số `123` hoặc `26 - (ord - 97)`)?
- [ ] Có nhân với vị trí, rồi cộng tất cả?
- [ ] Đã thử `"abc"` = 148, `"zaza"` = 160, `n = 1`?
- [ ] Chữ ký đúng `reverseDegree(self, s: str) -> int`?